#### The objective of this process is to provide necessary data to respond to the research question:
***Where is an escalation of violence predicted***

In [8]:
import os
import sys

# Get the current working directory
current_directory = os.getcwd()

# Print the current working directory
print("The current Working Directory is:", current_directory)

# Get the path to the base directory (VIEWS_FAO_index)
base_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
print(f'The base directory will be set to: {base_dir}')

# Add the base directory to sys.path
sys.path.insert(0, base_dir)

The current Working Directory is: /Users/gbenz/Documents/VIEWS_FAO_index/notebooks/methods
The base directory will be set to: /Users/gbenz/Documents/VIEWS_FAO_index


In [9]:
# delte this is temporary--
from src.utils.functions_for_graphics.individual_graphics.map_helper.manipulate_tables_for_mapping import calculate_histogram_data


from src.utils.universal_functions.setup.generate_base_file import give_primary_frame
from src.utils.universal_functions.setup.generate_percentiles import get_percentiles
from src.utils.universal_functions.setup.define_scale import select_scale

from src.utils.functions_for_return_periods.insurance_products_for_RP import insurance_files

from src.utils.universal_functions.setup.build_directory import float_to_custom_string, ensure_directory_exists


#Functions for graphics:
from src.utils.functions_for_graphics.construct_custom_symbology import user_defined_colormap, display_percentile_color_table, assign_color_to_zero_values, process_and_merge_return_periods


from src.utils.functions_for_graphics.individual_graphics.map_helper.manipulate_tables_for_mapping import clean_info_dataframe, query_and_sort_annual_table, provide_values_at_input_return_periods, retrieve_geodataframe, define_year_to_map, query_geodataframe

from src.utils.functions_for_graphics.individual_graphics.image_annual_returnperiod_table import image_save_returnperiodtable
from src.utils.functions_for_graphics.individual_graphics.image_annual_returnperiod_lineplot import plot_histogram_with_lineplot_4
from src.utils.functions_for_graphics.individual_graphics.image_annual_summary_table import plot_and_colorize_annual_table
from src.utils.functions_for_graphics.individual_graphics.image_map_for_Ei import image_save_map_E_i

#Mapping structure:
from src.utils.functions_for_graphics.layout_formats.event_cat_rp import map_event_cat_rp
from src.utils.functions_for_graphics.layout_formats.summary_of_top_years import map_top_years
#from src.utils.functions_for_graphics.layout_formats.Layout_single_method_option1 import mapped_option1


#functions for all methods
from src.utils.universal_functions.FAO_table_formatting.generate_output_tables import generate_and_give_info_dataframe, append_return_periods_to_annual_table

The current Working Directory is: /Users/gbenz/Documents/VIEWS_FAO_index/notebooks/methods
The base directory will be set to: /Users/gbenz/Documents/VIEWS_FAO_index
The current Working Directory is: /Users/gbenz/Documents/VIEWS_FAO_index/notebooks/methods
The base directory will be set to: /Users/gbenz/Documents/VIEWS_FAO_index
The current Working Directory is: /Users/gbenz/Documents/VIEWS_FAO_index/notebooks/methods
The base directory will be set to: /Users/gbenz/Documents/VIEWS_FAO_index
The current Working Directory is: /Users/gbenz/Documents/VIEWS_FAO_index/notebooks/methods
The base directory will be set to: /Users/gbenz/Documents/VIEWS_FAO_index
The current Working Directory is: /Users/gbenz/Documents/VIEWS_FAO_index/notebooks/methods
The base directory will be set to: /Users/gbenz/Documents/VIEWS_FAO_index
The current Working Directory is: /Users/gbenz/Documents/VIEWS_FAO_index/notebooks/methods
The base directory will be set to: /Users/gbenz/Documents/VIEWS_FAO_index
The curren

## Construct ***Historical*** Dataframe
#### Provide user inputs to caputure a dataframe for `Recent Conflict History`

In [38]:
data = give_primary_frame('Fatalities_fao_pgm', 'cm_properties', 2023, 2026)

100%|██████████| 40.0M/40.0M [00:03<00:00, 11.7MB/s]


Queryset Fatalities_fao_pgm read successfully 
Queryset cm_properties read successfully 
['month_id', 'pg_id', 'C_start_year', 'C_end_year', 'pop_gpw_sum', 'country_name', 'ged_sb', 'ged_ns', 'ged_os', 'year', 'fatalities_sum', 'country_id']


#### Assign `month` and `date` columns

- Subsect state based conflict `ged_sb` that is not zero
- Identify the most recent date of reported conflict events and save to the variable `latest_date`

In [39]:
from ingester3.extensions import *
data['month'] = data.m.month

# Create a date column with the first day of each month
data['date'] = pd.to_datetime(
    data['year'].astype(str) + '-' + data['month'].astype(str).str.zfill(2) + '-01'
)
# Filter rows where ged_sb is non-zero
mask = data['ged_sb'] != 0
latest_date = data.loc[mask, 'date'].max()

print("Latest date with non-zero ged_sb:", latest_date)

Latest date with non-zero ged_sb: 2025-01-01 00:00:00


## Reference VIEWS ***Predictions*** Dataframe

In [19]:
import pandas as pd

# Define the URL of the CSV file
url = 'https://viewsforecasting.org/wp-content/uploads/fatalities002_2024_12_t01_pgm.csv'

# Read the CSV file into a DataFrame
predictions = pd.read_csv(url)
# Optional: Display the first few rows to verify

predictions['month'] = predictions.m.month

# Extract the month as a two-digit string and the year as an integer
predictions['month'] = predictions['month'].astype(int).astype(str).str.zfill(2)
predictions['year'] = predictions.m.year

# Create a date column with the first day of each month
predictions['date'] = pd.to_datetime(
    predictions['year'].astype(str) + '-' + predictions['month'].astype(str).str.zfill(2) + '-01'
)

print(predictions.head())


   pg_id  month_id  main_mean_ln  main_mean  main_dich month  year       date
0  62356       541        0.0001     0.0001     0.0062    01  2025 2025-01-01
1  79599       541        0.0001     0.0001     0.0062    01  2025 2025-01-01
2  79600       541        0.0002     0.0002     0.0062    01  2025 2025-01-01
3  79601       541        0.0002     0.0002     0.0062    01  2025 2025-01-01
4  80317       541        0.0002     0.0002     0.0062    01  2025 2025-01-01


***Reference the date from the predictions dataframe for one month and one year ahead***

In [71]:
import numpy as np

# First, get the unique dates (ensuring they're in datetime format)
unique_dates = pd.to_datetime(predictions['date'].unique())

# Sort the dates; np.sort returns a sorted numpy array
sorted_dates = np.sort(unique_dates)

# The most recent date will be the last element in the sorted array
most_recent_date = sorted_dates[0]
year_prediction = sorted_dates[11]

print("Most recent date:", most_recent_date)
print("One year prediction date:", year_prediction)

subset_one_month_predictions = predictions[predictions['date'] == most_recent_date]

subset_one_year_predictions = predictions[predictions['date'] == year_prediction]



Most recent date: 2025-01-01T00:00:00.000000000
One year prediction date: 2025-12-01T00:00:00.000000000


In [ ]:
# Export dataframes to csv files:
subset_one_month_predictions.to_csv('/Users/gbenz/Documents/VIEWS_FAO_index/data/processed/one_month_predictions.csv')
subset_one_year_predictions.to_csv('/Users/gbenz/Documents/VIEWS_FAO_index/data/processed/one_year_predictions.csv')

#### Having identified the date derived from the most recent VIEWS Predictions
***Reference the date from recent historical data for one, three, and six months prior to the predictions***


In [72]:
import pandas as pd

# Define your start time variable (replacing latest_date)
start_date = pd.to_datetime(most_recent_date)

# Subtract 1, 3, and 6 months using DateOffset
date_1_month_behind = start_date - pd.DateOffset(months=1)
date_3_months_behind = start_date - pd.DateOffset(months=3)
date_6_months_behind = start_date - pd.DateOffset(months=6)

# Display the computed dates
print("1 month behind:", date_1_month_behind)
print("3 months behind:", date_3_months_behind)
print("6 months behind:", date_6_months_behind)

# Now, subset the DataFrame for each computed date
subset_one_month_recent = data[data['date'] == date_1_month_behind]
subset_three_month_recent = data[data['date'] == date_3_months_behind]
subset_six_month_recent = data[data['date'] == date_6_months_behind]

1 month behind: 2024-12-01 00:00:00
3 months behind: 2024-10-01 00:00:00
6 months behind: 2024-07-01 00:00:00


In [ ]:
# Export dataframes to csv files:
subset_one_month_recent.to_csv('/Users/gbenz/Documents/VIEWS_FAO_index/data/processed/one_month_recent.csv')
subset_three_month_recent.to_csv('/Users/gbenz/Documents/VIEWS_FAO_index/data/processed/three_month_recent.csv')
subset_six_month_recent.to_csv('/Users/gbenz/Documents/VIEWS_FAO_index/data/processed/six_month_recent.csv')

### Evaluate ***Escalation***
#### Here we will difference the ***Predictions*** from the ***observed recent conflict history***

In [78]:
# Define the prediction field to evaluate ['main_mean','main_mean_ln', 'main_dich']

prediction_field = 'main_mean'

In [81]:
# ----------------------------------------------------------------------------------------------------------
# Compare one month prediction to most recent observed historical month
# ----------------------------------------------------------------------------------------------------------
one_month_rec__one_month_pred = pd.merge(subset_one_month_recent, subset_one_month_predictions, on='pg_id')

one_month_rec__one_month_pred['one_month_dif'] =  one_month_rec__one_month_pred[prediction_field] - one_month_rec__one_month_pred['ged_sb'] 
# ----------------------------------------------------------------------------------------------------------
# Define necessary columns from 'one_month_rec__one_month_pred':
# ----------------------------------------------------------------------------------------------------------
one_month_rec__one_month_pred__organized = one_month_rec__one_month_pred[['pg_id', 'ged_sb', prediction_field, 'one_month_dif']]
# ----------------------------------------------------------------------------------------------------------
# ----------------------------------------------------------------------------------------------------------


# ----------------------------------------------------------------------------------------------------------
# Compare one year prediction to most recent observed historical month
# ----------------------------------------------------------------------------------------------------------
one_month_rec__one_year_pred = pd.merge(subset_one_month_recent, subset_one_year_predictions, on='pg_id')

one_month_rec__one_year_pred['one_year_dif'] = one_month_rec__one_year_pred[prediction_field]- one_month_rec__one_year_pred['ged_sb'] 
# ----------------------------------------------------------------------------------------------------------
# Define necessary columns from 'one_month_rec__one_month_pred':
# ----------------------------------------------------------------------------------------------------------
one_month_rec__one_year_pred__organized = one_month_rec__one_year_pred[['pg_id', 'one_year_dif']]
# ----------------------------------------------------------------------------------------------------------
# ----------------------------------------------------------------------------------------------------------


# ----------------------------------------------------------------------------------------------------------
# Merge predicted one month with one year
# ----------------------------------------------------------------------------------------------------------
combined_predictions = pd.merge(one_month_rec__one_year_pred__organized, one_month_rec__one_month_pred__organized, on='pg_id')
# ----------------------------------------------------------------------------------------------------------
display(combined_predictions.head(10))
# ----------------------------------------------------------------------------------------------------------


# ----------------------------------------------------------------------------------------------------------
# Country Year Summary for one month predicted change
# ----------------------------------------------------------------------------------------------------------
cm_one_month = one_month_rec__one_month_pred.groupby('country_name', as_index=False)[['one_month_dif', 'ged_sb', prediction_field]].sum()
cm_one_year = one_month_rec__one_year_pred.groupby('country_name', as_index=False)['one_year_dif'].sum()
# ----------------------------------------------------------------------------------------------------------
# Define necessary columns from 'one_month_rec__one_month_pred':
# ----------------------------------------------------------------------------------------------------------
combined_country = pd.merge(cm_one_year, cm_one_month, on='country_name')
# ----------------------------------------------------------------------------------------------------------
combined_country = combined_country[['country_name', 'ged_sb', prediction_field, 'one_month_dif', 'one_year_dif']]
# ----------------------------------------------------------------------------------------------------------
# ----------------------------------------------------------------------------------------------------------
display(combined_country.head(100))



,pg_id,one_year_dif,ged_sb,main_mean,one_month_dif
0,62356,0.0002,0.0,0.0001,0.0001
1,79599,0.0003,0.0,0.0001,0.0001
2,79600,0.0004,0.0,0.0002,0.0002
3,79601,0.0003,0.0,0.0002,0.0002
4,80317,0.0005,0.0,0.0002,0.0002
5,80318,0.0005,0.0,0.0005,0.0005
6,80319,0.0003,0.0,0.0001,0.0001
7,80320,0.0003,0.0,0.0002,0.0002
8,80321,0.0004,0.0,0.0001,0.0001
9,80322,0.0004,0.0,0.0001,0.0001


,country_name,ged_sb,main_mean,one_month_dif,one_year_dif
0,Afghanistan,0.0,31.7924,31.7924,62.6568
1,Algeria,0.0,0.1713,0.1713,0.2766
2,Angola,0.0,0.2853,0.2853,0.4283
3,Armenia,0.0,0.1159,0.1159,0.3442
4,Azerbaijan,0.0,0.1925,0.1925,0.6412
5,Bahrain,0.0,0.0113,0.0113,0.0143
6,Benin,15.0,3.3144,-11.6856,-10.8443
7,Botswana,0.0,0.0000,0.0000,0.0027
8,Bulgaria,0.0,0.0014,0.0014,0.0178
9,Burkina Faso,27.0,75.6536,48.6536,104.0333


In [82]:
# Export dataframes to csv files:

one_month_rec__one_month_pred.to_csv('/Users/gbenz/Documents/VIEWS_FAO_index/data/processed/one_month_change.csv')
one_month_rec__one_year_pred.to_csv('/Users/gbenz/Documents/VIEWS_FAO_index/data/processed/one_year_change.csv')

combined_predictions.to_csv('/Users/gbenz/Documents/VIEWS_FAO_index/data/processed/combined_predictions.csv')
combined_country.to_csv('/Users/gbenz/Documents/VIEWS_FAO_index/data/processed/country_change.csv')
